# Klasik Görüntü İşleme ile Hücre Segmentasyonu, Sayımı ve Zamansal Analizi

## Bölüm 2 – Veri Setinin İndirilmesi ve Ön İncelemesi

In [ ]:
# Bölüm 2.1–2.2: Veri seti klasörlerini ve sekansları inceleme
from pathlib import Path

VERI = Path("data/Fluo-N2DH-GOWT1")

for klasor in sorted(VERI.iterdir()):
    tif_sayisi = len(list(klasor.rglob("*.tif")))
    print(f"{klasor.name:12s} -> {tif_sayisi} tif dosyası")

In [ ]:
# Bölüm 2.3: Görüntülerin boyutu, veri tipi ve piksel yoğunlukları
#
# Test sızıntısını önlemek için:
#   - Sekans 01 (geliştirme + doğrulama havuzu): boyut, tip ve yoğunluklar detaylı incelenir.
#   - Sekans 02 (test havuzu): yalnızca kare sayısı, boyut ve veri tipi kontrol edilir.
#     Yoğunluk/histogram bilgisine bakılmaz; yöntem seçimini etkilememesi için.
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt

def goruntu_oku(yol):
    """TIFF dosyasını orijinal bit derinliğiyle okur.
    cv2.imread Türkçe karakterli yollarda çalışmadığı için
    dosya önce bayt olarak okunur, sonra cv2.imdecode ile çözülür."""
    veri = np.fromfile(str(yol), dtype=np.uint8)
    return cv2.imdecode(veri, cv2.IMREAD_UNCHANGED)

# --- Sekans 02: yalnızca biçim kontrolü ---
dosyalar_02 = sorted((VERI / "02").glob("t*.tif"))
bicimler_02 = {(goruntu_oku(d).shape, str(goruntu_oku(d).dtype)) for d in dosyalar_02}
print(f"Sekans 02: {len(dosyalar_02)} kare, (boyut, tip) = {bicimler_02}")

# --- Sekans 01: detaylı inceleme ---
satirlar = []
for dosya in sorted((VERI / "01").glob("t*.tif")):
    g = goruntu_oku(dosya)
    satirlar.append({
        "kare": int(dosya.stem[1:]),          # "t042" -> 42
        "boyut": f"{g.shape[0]}x{g.shape[1]}",
        "tip": str(g.dtype),
        "min": int(g.min()),
        "max": int(g.max()),
        "ortalama": g.mean(),
        "p99": np.percentile(g, 99),          # piksellerin %99'u bu değerin altında
        "sifir_orani": (g == 0).mean(),
    })

istatistik = pd.DataFrame(satirlar)
print(f"Sekans 01: {len(istatistik)} kare, boyutlar = {istatistik.boyut.unique()}, tipler = {istatistik.tip.unique()}")

# Sekans 01 sayısal özeti (kareler arası dağılım)
istatistik[["min", "max", "ortalama", "p99", "sifir_orani"]].describe().round(3)

In [ ]:
# Bölüm 2.3: Histogram özellikleri - Sekans 01'in TÜM karelerinin birleşik histogramı
toplam = np.zeros(256)
for dosya in sorted((VERI / "01").glob("t*.tif")):
    toplam += np.bincount(goruntu_oku(dosya).ravel(), minlength=256)
hist_01 = toplam / toplam.sum()               # oran (0-1) olarak normalize

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(np.arange(256), hist_01, label="Sekans 01 (92 kare)")
ax.set_yscale("log")                           # arka plan tepesi diğer değerleri ezmesin
ax.set_xlabel("Piksel yoğunluğu")
ax.set_ylabel("Piksel oranı (log)")
ax.set_title("Sekans 01 - tüm karelerin birleşik histogramı")
ax.legend()
ax.grid(alpha=0.3)
plt.show()

# Sayısal özet: yoğunluğun hangi aralıkta toplandığı
kum = np.cumsum(hist_01)
for oran in [0.50, 0.90, 0.99, 0.999]:
    print(f"Piksellerin %{oran*100:g}'i <= {np.searchsorted(kum, oran)}")

In [ ]:
# Bölüm 2.3: Sekans 01'de parlaklığın kare kare (zamanla) değişimi
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
ax[0].plot(istatistik.kare, istatistik.ortalama)
ax[1].plot(istatistik.kare, istatistik.p99)
ax[0].set_title("Sekans 01 - kare ortalama yoğunluğu")
ax[1].set_title("Sekans 01 - kare 99. yüzdelik yoğunluğu (parlak bölgeler)")
for a in ax:
    a.set_xlabel("Kare numarası")
    a.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Bölüm 2.4: Hücrelerin şekil, boyut ve parlaklık farklılıkları
#
# 01_GT/SEG klasöründeki manuel segmentasyon maskeleri hücre sınırlarını hassas biçimde verir.
# (Bu maskeler her karede tüm hücreleri değil, yalnızca bir kısmını içerir; ama ölçüm için yeterli.)
# Her etiketli hücre için alan, eşdeğer çap, şekil ve ortalama parlaklık ölçülür.
from skimage.measure import regionprops_table

SEG_KLASOR = VERI / "01_GT" / "SEG"
hucre_satirlari = []
for maske_yolu in sorted(SEG_KLASOR.glob("man_seg*.tif")):
    kare_no = int(maske_yolu.stem.replace("man_seg", ""))     # "man_seg014" -> 14
    maske = goruntu_oku(maske_yolu)                            # her hücre farklı bir etiket (1, 2, ...)
    goruntu = goruntu_oku(VERI / "01" / f"t{kare_no:03d}.tif")
    olcum = regionprops_table(
        maske, intensity_image=goruntu,
        properties=["label", "area", "equivalent_diameter_area", "eccentricity",
                    "solidity", "intensity_mean", "intensity_min", "intensity_max"])
    tablo = pd.DataFrame(olcum)
    tablo["kare"] = kare_no
    hucre_satirlari.append(tablo)

hucreler = pd.concat(hucre_satirlari, ignore_index=True)
print(f"{hucreler.kare.nunique()} SEG karesinden toplam {len(hucreler)} hücre ölçüldü.\n")
print("eccentricity: 0 = tam daire, 1'e yaklaştıkça uzun/elips")
print("solidity    : 1 = tamamen dışbükey, düşük değer = girintili/düzensiz şekil\n")

# Ölçümlerin dağılımı
fig, ax = plt.subplots(1, 4, figsize=(18, 3.5))
ax[0].hist(hucreler.area, bins=25);                      ax[0].set_title("Alan (piksel)")
ax[1].hist(hucreler.equivalent_diameter_area, bins=25);  ax[1].set_title("Eşdeğer çap (piksel)")
ax[2].hist(hucreler.eccentricity, bins=25);              ax[2].set_title("Eccentricity")
ax[3].hist(hucreler.intensity_mean, bins=25);            ax[3].set_title("Ortalama parlaklık")
for a in ax:
    a.set_ylabel("Hücre sayısı")
    a.grid(alpha=0.3)
plt.tight_layout()
plt.show()

hucreler[["area", "equivalent_diameter_area", "eccentricity", "solidity",
          "intensity_mean", "intensity_min", "intensity_max"]].describe().round(2)

In [ ]:
# Bölüm 2.5 (a): Olası güçlüklerin görsel incelenmesi
#
# Ham görüntüler çok karanlık olduğu için (piksellerin %90'ı <= 6) YALNIZCA GÖSTERİM AMACIYLA
# kontrast artırılır: [0, p99.9] aralığına kırpılır ve gamma=0.5 uygulanır (soluk bölgeler açılır).
# Bu işlem segmentasyonda kullanılmaz.
# Kare seçimi keyfi olmasın diye sekansın başı, ortası ve sonu alınır.
# Kırmızı çizgiler: 01_GT/TRA takip etiketleri (referansa göre hücrelerin yeri).
from skimage.segmentation import find_boundaries

def gosterim_icin(g, ust_yuzdelik=99.9, gamma=0.5):
    ust = np.percentile(g, ust_yuzdelik)
    return (np.clip(g, 0, ust) / ust) ** gamma

ornek_kareler = [0, 45, 91]
fig, ax = plt.subplots(2, 3, figsize=(18, 12))
for i, kare_no in enumerate(ornek_kareler):
    g = goruntu_oku(VERI / "01" / f"t{kare_no:03d}.tif")
    tra = goruntu_oku(VERI / "01_GT" / "TRA" / f"man_track{kare_no:03d}.tif")

    ax[0, i].imshow(g, cmap="gray", vmin=0, vmax=255)
    ax[0, i].set_title(f"t{kare_no:03d} - ham görüntü (0-255)")

    renkli = np.dstack([gosterim_icin(g)] * 3)
    renkli[find_boundaries(tra, mode="outer")] = [1, 0, 0]
    ax[1, i].imshow(renkli)
    ax[1, i].set_title(f"t{kare_no:03d} - gösterim için kontrast + TRA ({len(np.unique(tra)) - 1} hücre)")
for a in ax.ravel():
    a.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# Bölüm 2.5 (b): Güçlüklerin sayısal olarak ölçülmesi (Sekans 01, 92 karenin tamamı)
#
# 1) Temas eden / çok yakın hücreler : TRA etiketleri 10 piksel genişletilince birleşen hücre sayısı
#    (TRA etiketleri gerçek hücreden biraz küçük olduğu için genişletme gerçek sınıra yaklaştırır)
# 2) Kontrast yetersizliği             : ortalama parlaklığı arka plana yakın olan hücrelerin oranı (Bölüm 2.4)
# 3) Gürültü                           : hücrelerin dışında kalan küçük parlak lekeler (>= 50 yoğunluk, < 100 piksel)
# 4) Düzensiz arka plan                : hücre dışı piksellerin 8x8 bloklardaki ortalaması
cekirdek = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (21, 21))   # yarıçap 10 piksel
kare_satirlari = []
blok_haritalari = []
for kare_no in range(92):
    g = goruntu_oku(VERI / "01" / f"t{kare_no:03d}.tif")
    tra = goruntu_oku(VERI / "01_GT" / "TRA" / f"man_track{kare_no:03d}.tif")
    hucre_sayisi = len(np.unique(tra)) - 1

    genis = cv2.dilate((tra > 0).astype(np.uint8), cekirdek)
    bilesen_sayisi = cv2.connectedComponents(genis)[0] - 1          # 0 = arka plan

    hucre_disi = cv2.dilate(genis, cekirdek) == 0                   # hücrelerden emin olunacak kadar uzak bölge
    parlak = ((g >= 50) & hucre_disi).astype(np.uint8)
    n, _, stats, _ = cv2.connectedComponentsWithStats(parlak)
    kucuk_leke = int(np.sum(stats[1:, cv2.CC_STAT_AREA] < 100))

    bloklar = np.full((8, 8), np.nan)
    for r in range(8):
        for c in range(8):
            dilim = (slice(r * 128, (r + 1) * 128), slice(c * 128, (c + 1) * 128))
            if hucre_disi[dilim].sum() > 1000:
                bloklar[r, c] = g[dilim][hucre_disi[dilim]].mean()
    blok_haritalari.append(bloklar)

    kare_satirlari.append({"kare": kare_no, "hucre": hucre_sayisi,
                           "birlesen_hucre": hucre_sayisi - bilesen_sayisi,
                           "kucuk_parlak_leke": kucuk_leke,
                           "arka_plan_blok_min": np.nanmin(bloklar),
                           "arka_plan_blok_max": np.nanmax(bloklar)})

guclukler = pd.DataFrame(kare_satirlari)

print("1) Temas eden / çok yakın hücreler")
print(f"   Karelerin %{(guclukler.birlesen_hucre > 0).mean()*100:.0f}'inde en az bir birleşme var; "
      f"kare başına ortalama {guclukler.birlesen_hucre.mean():.2f}, en fazla {guclukler.birlesen_hucre.max()}")
print("2) Kontrast yetersizliği (SEG hücreleri)")
print(f"   Ortalama parlaklığı <= 6 olan (arka planın %90'ıyla aynı düzeyde) hücreler: "
      f"%{(hucreler.intensity_mean <= 6).mean()*100:.0f}; <= 10 olanlar: %{(hucreler.intensity_mean <= 10).mean()*100:.0f}")
print("3) Gürültü")
print(f"   Hücre dışında kare başına ortalama {guclukler.kucuk_parlak_leke.mean():.1f} küçük parlak leke "
      f"(en fazla {guclukler.kucuk_parlak_leke.max()})")
print("4) Düzensiz arka plan")
print(f"   Hücre dışı 8x8 blok ortalamaları: en düşük {guclukler.arka_plan_blok_min.mean():.2f}, "
      f"en yüksek {guclukler.arka_plan_blok_max.mean():.2f} (92 karenin ortalaması)")

fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
ax[0].plot(guclukler.kare, guclukler.hucre, label="Referans hücre sayısı (TRA)")
ax[0].plot(guclukler.kare, guclukler.hucre - guclukler.birlesen_hucre, label="Genişletince ayrı kalan bileşen")
ax[0].set_xlabel("Kare numarası"); ax[0].set_title("Temas eden hücrelerin sayıma etkisi")
ax[0].legend(); ax[0].grid(alpha=0.3)
ort_harita = np.nanmean(blok_haritalari, axis=0)
im = ax[1].imshow(ort_harita, cmap="viridis")
ax[1].set_title("Hücre dışı arka plan - 8x8 blok ortalaması (92 kare)")
plt.colorbar(im, ax=ax[1])
plt.tight_layout()
plt.show()

## Bölüm 3 – Geliştirme, Doğrulama ve Test Görüntülerinin Seçilmesi

- Seed = öğrenci numarasının son 5 hanesi + 2026 = 29015 + 2026 = **31041**
- Sekans 01: 5 doğrulama + 10 geliştirme karesi (çakışmaz, aralarında en az 5 kare boşluk)
- Sekans 02: 3 test bloğu × ardışık 5 kare (çakışmaz, aralarında en az 5 kare boşluk)
- "En az 5 karelik boşluk": iki kare arasında en az 5 kare bulunmalı, yani kare numarası farkı ≥ 6.
- Koşullar sağlanmazsa aynı RNG akışından yeniden çekilir (deterministik tekrar).
- Seçim, algoritma geliştirilmeden önce yapılır ve sonradan değiştirilmez.

In [ ]:
# Bölüm 3: Rastgele ve tekrarlanabilir görüntü seçimi -> selected_frames.csv
# Bu hücre referans (GT) dosyalarını OKUMAZ; yalnızca kare numaralarını belirler.
OGRENCI_NO_SON5 = 29015
SEED = OGRENCI_NO_SON5 + 2026          # 31041

KARE_SAYISI = 92
DOGRULAMA_SAYISI = 5
GELISTIRME_SAYISI = 10
TEST_BLOK_SAYISI = 3
BLOK_UZUNLUGU = 5
MIN_BOSLUK = 5                         # iki kare arasında bulunması gereken en az kare sayısı
MAX_DENEME = 1000

def yeterince_uzak(a, b):
    """a ile b arasında en az MIN_BOSLUK kare varsa True (ör. 20 ile 26 -> arada 5 kare)."""
    return abs(a - b) - 1 >= MIN_BOSLUK

def sekans01_sec(rng):
    """Önce 5 doğrulama karesi, sonra onlardan yeterince uzak 10 geliştirme karesi seçer."""
    for deneme in range(1, MAX_DENEME + 1):
        dogrulama = np.sort(rng.choice(KARE_SAYISI, size=DOGRULAMA_SAYISI, replace=False))
        uygun = [k for k in range(KARE_SAYISI) if all(yeterince_uzak(k, d) for d in dogrulama)]
        if len(uygun) >= GELISTIRME_SAYISI:
            gelistirme = np.sort(rng.choice(uygun, size=GELISTIRME_SAYISI, replace=False))
            return dogrulama.tolist(), gelistirme.tolist(), deneme
    raise RuntimeError("Sekans 01 için koşulları sağlayan seçim bulunamadı")

def sekans02_sec(rng):
    """Birbirinden en az MIN_BOSLUK kare uzak 3 adet, 5 karelik ardışık blok seçer."""
    son_baslangic = KARE_SAYISI - BLOK_UZUNLUGU         # 87 -> blok 87..91
    for deneme in range(1, MAX_DENEME + 1):
        baslangiclar = np.sort(rng.choice(son_baslangic + 1, size=TEST_BLOK_SAYISI, replace=False))
        uygun_mu = all(baslangiclar[i + 1] - (baslangiclar[i] + BLOK_UZUNLUGU - 1) - 1 >= MIN_BOSLUK
                       for i in range(TEST_BLOK_SAYISI - 1))
        if uygun_mu:
            return [list(range(b, b + BLOK_UZUNLUGU)) for b in baslangiclar], deneme
    raise RuntimeError("Sekans 02 için koşulları sağlayan blok seçimi bulunamadı")

rng = np.random.default_rng(SEED)      # tek RNG akışı: aynı sıra -> her çalıştırmada aynı sonuç
dogrulama, gelistirme, deneme01 = sekans01_sec(rng)
test_bloklari, deneme02 = sekans02_sec(rng)

secim_satirlari = []
for k in gelistirme:
    secim_satirlari.append(("development", "01", k, ""))
for k in dogrulama:
    secim_satirlari.append(("validation", "01", k, ""))
for blok_no, kareler in enumerate(test_bloklari, start=1):
    for k in kareler:
        secim_satirlari.append(("test", "02", k, blok_no))

secim = pd.DataFrame(secim_satirlari, columns=["group", "sequence", "frame", "test_block"])
secim["image_file"] = [f"{s}/t{k:03d}.tif" for s, k in zip(secim.sequence, secim.frame)]
secim["reference_file"] = [f"{s}_GT/TRA/man_track{k:03d}.tif" for s, k in zip(secim.sequence, secim.frame)]
secim["seed"] = SEED

# Kural kontrolleri
assert len(secim) == 30
assert not set(gelistirme) & set(dogrulama)
assert all(yeterince_uzak(g, d) for g in gelistirme for d in dogrulama)
assert all((VERI / p).exists() for p in secim.image_file)     # yalnızca dosya var mı; içerik okunmaz

secim.to_csv("selected_frames.csv", index=False)

print(f"Seed: {SEED}")
print(f"Deneme sayısı -> Sekans 01: {deneme01}, Sekans 02: {deneme02}")
print(f"Doğrulama  (01): {dogrulama}")
print(f"Geliştirme (01): {gelistirme}")
for blok_no, kareler in enumerate(test_bloklari, start=1):
    print(f"Test blok {blok_no} (02): {kareler}")
print("Kaydedildi: selected_frames.csv")

# Seçimin zaman ekseninde görünümü
fig, ax = plt.subplots(figsize=(14, 2.2))
ax.scatter(gelistirme, [1] * len(gelistirme), marker="s", s=60, label="Geliştirme (01)")
ax.scatter(dogrulama, [1] * len(dogrulama), marker="D", s=60, label="Doğrulama (01)")
for kareler in test_bloklari:
    ax.scatter(kareler, [0] * len(kareler), marker="o", s=60, color="tab:red")
ax.scatter([], [], marker="o", color="tab:red", label="Test blokları (02)")
ax.set_yticks([0, 1]); ax.set_yticklabels(["Sekans 02", "Sekans 01"])
ax.set_xlim(-1, 92); ax.set_ylim(-0.6, 1.6)
ax.set_xlabel("Kare numarası"); ax.grid(alpha=0.3)
ax.legend(loc="upper center", bbox_to_anchor=(0.5, 1.45), ncol=3)
plt.show()

## Bölüm 4 – Referans Hücre Sayılarının Belirlenmesi

- Referans: `01_GT/TRA/man_trackXXX.tif` takip etiketleri. Her hücre farklı bir pozitif kimlikle (1, 2, 3, ...) işaretlidir; 0 arka plandır.
- Referans sayı = sıfır hariç birbirinden farklı kimliklerin sayısı.
- **Yalnızca geliştirme (10) ve doğrulama (5) karelerinin** referansları okunur.
- Test referansları (`02_GT/TRA`) nihai pipeline ve test tahminleri kilitlenip commit edilene kadar okunmaz; kod bunu bir kilit değişkeniyle engeller.

In [ ]:
# Bölüm 4: Geliştirme ve doğrulama karelerinin referans hücre sayıları
from scipy import ndimage

# Bölüm 12'de tahminler kilitlenip commit edilene kadar True kalacak
TEST_REFERANSLARI_KILITLI = True

def referans_oku(satir):
    """selected_frames.csv'deki bir satırın TRA etiket görüntüsünü okur.
    Test satırları kilit açılmadan okunamaz."""
    if satir.group == "test" and TEST_REFERANSLARI_KILITLI:
        raise PermissionError("Test referansları kilitli! Önce Bölüm 12'deki kilitleme tamamlanmalı.")
    return goruntu_oku(VERI / satir.reference_file)

ref_satirlari = []
for satir in secim[secim.group != "test"].itertuples():
    tra = referans_oku(satir)
    kimlikler = np.unique(tra)
    kimlikler = kimlikler[kimlikler != 0]                     # 0 = arka plan, sayılmaz

    # Belirsizlik kontrolleri (doküman madde 4):
    # - kenara değen hücre: görüntü sınırında kesilmiş olabilir
    # - parçalı etiket: aynı kimlik birden fazla kopuk parçadan oluşuyor mu
    kenar = np.unique(np.concatenate([tra[0], tra[-1], tra[:, 0], tra[:, -1]]))
    kenara_degen = int(np.sum(kenar != 0))
    parcali = sum(ndimage.label(tra == k)[1] > 1 for k in kimlikler)

    ref_satirlari.append({"group": satir.group, "frame": satir.frame,
                          "reference_count": len(kimlikler),
                          "kenara_degen": kenara_degen, "parcali_etiket": int(parcali)})

referanslar = pd.DataFrame(ref_satirlari)
print(referanslar.to_string(index=False))
print()
print(referanslar.groupby("group").reference_count.agg(["count", "mean", "min", "max"]).round(2))

In [ ]:
# Bölüm 4 (madde 5): Referans sayılarının görsel kontrolü - 15 geliştirme + doğrulama karesi
# Kontrast yalnızca gösterim için artırılır (Bölüm 2.5'teki gosterim_icin fonksiyonu).
# Kırmızı: TRA etiketlerinin sınırı. Her etiketin ortasına kimlik numarası yazılır.
gelistirme_dogrulama = secim[secim.group != "test"].reset_index(drop=True)
fig, ax = plt.subplots(3, 5, figsize=(25, 15.5))
for a, satir in zip(ax.ravel(), gelistirme_dogrulama.itertuples()):
    g = goruntu_oku(VERI / satir.image_file)
    tra = referans_oku(satir)
    renkli = np.dstack([gosterim_icin(g)] * 3)
    renkli[find_boundaries(tra, mode="outer")] = [1, 0, 0]
    a.imshow(renkli)
    for k in np.unique(tra)[1:]:
        y, x = np.argwhere(tra == k).mean(axis=0)
        a.text(x, y, str(k), color="yellow", fontsize=7, ha="center", va="center")
    sayi = referanslar.loc[referanslar.frame == satir.frame, "reference_count"].item()
    grup = "geliştirme" if satir.group == "development" else "doğrulama"
    a.set_title(f"t{satir.frame:03d} ({grup}) - referans: {sayi} hücre")
    a.axis("off")
plt.tight_layout()
plt.show()

### Bölüm 4 – Görsel kontrol gözlemleri

1. **İşaretlenmemiş kenar hücresi:** Sol kenarda (x≈10, y≈320) görüntü sınırıyla kesilmiş soluk (ort. ≈8) bir hücre 15 karenin neredeyse hepsinde görülür ama TRA'da etiketi yoktur. Benzer biçimde t014, t078, t085'te üst kenarda kısmen görünen nesneler etiketsizdir. t020'de iç bölgede de etiketsiz soluk bir nesne vardır. Bu nesneleri bulan bir yöntem, referansa göre "fazla sayım" yapmış görünür.
2. **t020'deki küçük nesne (etiket 22) gerçek bir hücredir:** `man_track.txt`'e göre hücre 16, 16. karede bölünmüş; yavru hücreler 22 (17. kareden itibaren) ve 18'dir (21. kareden itibaren). Bölünmeden hemen sonra hücre küçük ve yoğundur; etiket alanı ≈250–400 pikselden başlayıp 50. kare civarında diğer hücrelerin boyutuna (≈1650) ulaşır. t085'teki 35 numaralı küçük nesne de hücre 23'ün yavrusudur.
3. **Sonuç:** Referans sayılar değiştirilmeden kullanılır. Kenar hücreleri ve yeni bölünmüş küçük hücreler hata analizinde ayrıca değerlendirilecektir. Minimum alan eşiği, yeni bölünmüş hücreleri silmeyecek kadar küçük, gürültü lekelerini (< 100 piksel) silecek kadar büyük seçilmelidir.

## Bölüm 5 – Başlangıç Yöntemi (Baseline)

Baseline, sonraki bütün yöntemlerin karşılaştırılacağı **basit referans yöntemdir**:

1. Gri seviyeli görüntüyü oku
2. Otsu eşikleme (tüm görüntü için tek, otomatik eşik)
3. Bağlı bileşen analizi (8-komşuluk)
4. Bileşenleri say (isteğe bağlı: sabit minimum alan eşiğinin altındakileri atla)
5. Sonucu görselleştir

Minimum alan eşiği **yalnızca geliştirme kareleri** üzerinde seçilir ve sonra sabitlenir. Baseline bundan sonra hiç değiştirilmez ve nihai yöntemle aynı test karelerinde karşılaştırılır.

**Metrikler** (gerçek = referans sayı, tahmin = yöntemin sayısı, hata = tahmin − gerçek):
- **MAE** = ortalama |hata| → ortalama kaç hücre yanılıyoruz
- **RMSE** = √(ortalama hata²) → büyük hataları daha çok cezalandırır
- **Bias** = ortalama hata → (+) fazla sayma, (−) eksik sayma eğilimi
- **Exact-count accuracy** = hatanın tam 0 olduğu karelerin oranı

In [ ]:
# Bölüm 5 (a): Baseline fonksiyonu, metrikler ve geliştirme kümesinde minimum alan seçimi
import time

def baseline_segment(g, min_alan=0):
    """Otsu eşikleme + 8-komşuluk bağlı bileşen analizi.
    Döndürür: (hücre sayısı, ikili maske, etiket görüntüsü, otsu eşiği)"""
    otsu_esik, maske = cv2.threshold(g, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    n, etiket, stats, _ = cv2.connectedComponentsWithStats(maske, connectivity=8)
    alanlar = stats[1:, cv2.CC_STAT_AREA]                  # 0 numaralı bileşen arka plan
    tutulan = np.flatnonzero(alanlar >= min_alan) + 1
    etiket = np.where(np.isin(etiket, tutulan), etiket, 0)
    return len(tutulan), maske, etiket, otsu_esik

def metrikler(tahmin, gercek):
    hata = np.asarray(tahmin) - np.asarray(gercek)
    return {"MAE": np.abs(hata).mean(),
            "RMSE": np.sqrt((hata ** 2).mean()),
            "Bias": hata.mean(),
            "Exact": (hata == 0).mean()}

# Geliştirme karelerini ve referanslarını bir kez belleğe al
gelistirme_kareleri = secim[secim.group == "development"].reset_index(drop=True)
dev_goruntuler = {s.frame: goruntu_oku(VERI / s.image_file) for s in gelistirme_kareleri.itertuples()}
dev_referans = referanslar.set_index("frame").reference_count

# Minimum alan eşiği taraması (yalnızca geliştirme kümesi)
ADAY_MIN_ALANLAR = [0, 10, 20, 30, 40, 50, 75, 100, 150, 200, 300, 500]
tarama = []
for min_alan in ADAY_MIN_ALANLAR:
    tahmin = [baseline_segment(dev_goruntuler[k], min_alan)[0] for k in dev_goruntuler]
    gercek = [dev_referans[k] for k in dev_goruntuler]
    tarama.append({"min_alan": min_alan, **metrikler(tahmin, gercek)})
tarama = pd.DataFrame(tarama)
print("Geliştirme kümesi - minimum alan eşiğine göre baseline başarımı:")
print(tarama.round(2).to_string(index=False))

BASELINE_MIN_ALAN = int(tarama.loc[tarama.MAE.idxmin(), "min_alan"])
print(f"\nSeçilen baseline minimum alanı: {BASELINE_MIN_ALAN} piksel (en düşük geliştirme MAE)")

fig, ax = plt.subplots(figsize=(9, 3.5))
ax.plot(tarama.min_alan, tarama.MAE, "o-", label="MAE")
ax.plot(tarama.min_alan, tarama.Bias, "s--", label="Bias")
ax.axhline(0, color="gray", lw=0.8)
ax.set_xscale("symlog", linthresh=10)
ax.set_xlabel("Minimum nesne alanı (piksel, symlog)"); ax.set_ylabel("Hücre")
ax.set_title("Baseline - minimum alan eşiğinin etkisi (geliştirme kümesi)")
ax.legend(); ax.grid(alpha=0.3)
plt.show()

# Sabitlenmiş baseline ile geliştirme kümesi sonuçları (kare bazında)
baseline_dev = []
for k, g in dev_goruntuler.items():
    t0 = time.perf_counter()
    sayi, maske, etiket, esik = baseline_segment(g, BASELINE_MIN_ALAN)
    sure = time.perf_counter() - t0
    baseline_dev.append({"frame": k, "reference": dev_referans[k], "baseline": sayi,
                         "hata": sayi - dev_referans[k], "otsu_esigi": esik,
                         "ham_bilesen": cv2.connectedComponents(maske, connectivity=8)[0] - 1,
                         "sure_ms": sure * 1000})
baseline_dev = pd.DataFrame(baseline_dev)
print(baseline_dev.round(1).to_string(index=False))
print({k: round(v, 3) for k, v in metrikler(baseline_dev.baseline, baseline_dev.reference).items()},
      f"| ortalama süre: {baseline_dev.sure_ms.mean():.1f} ms")

In [ ]:
# Bölüm 5 (b): Baseline segmentasyon çıktısının görselleştirilmesi
# Keyfi seçim olmasın diye: geliştirme kümesinde hatası EN BÜYÜK ve EN KÜÇÜK olan kareler gösterilir.
from skimage.color import label2rgb

en_kotu = baseline_dev.loc[baseline_dev.hata.abs().idxmax(), "frame"]
en_iyi = baseline_dev.loc[baseline_dev.hata.abs().idxmin(), "frame"]

fig, ax = plt.subplots(2, 3, figsize=(18, 12))
for satir_no, k in enumerate([en_kotu, en_iyi]):
    g = dev_goruntuler[k]
    tra = referans_oku(gelistirme_kareleri[gelistirme_kareleri.frame == k].iloc[0])
    sayi, maske, etiket, esik = baseline_segment(g, BASELINE_MIN_ALAN)

    renkli = np.dstack([gosterim_icin(g)] * 3)
    renkli[find_boundaries(tra, mode="outer")] = [1, 0, 0]
    ax[satir_no, 0].imshow(renkli)
    ax[satir_no, 0].set_title(f"t{k:03d} - görüntü + referans (kırmızı): {dev_referans[k]} hücre")

    ax[satir_no, 1].imshow(maske, cmap="gray")
    ax[satir_no, 1].set_title(f"Otsu maskesi (eşik = {esik:.0f})")

    ax[satir_no, 2].imshow(label2rgb(etiket, bg_label=0))
    ax[satir_no, 2].set_title(f"Bileşenler (alan >= {BASELINE_MIN_ALAN}): {sayi} nesne, hata {sayi - dev_referans[k]:+d}")
for a in ax.ravel():
    a.axis("off")
plt.tight_layout()
plt.show()

## Bölüm 6 – Yoğunluk Dönüşümlerinin Karşılaştırılması

**Kontrollü deney:** Yalnızca yoğunluk dönüşümü değiştirilir; geri kalan her şey baseline ile aynıdır (Otsu eşikleme → 8-komşuluk bağlı bileşen → minimum alan 50 piksel). Böylece sayımdaki değişim yalnızca dönüşüme bağlanabilir.

| ID | Dönüşüm | Parametre | Gerekçe |
|---|---|---|---|
| E01 | Orijinal (baseline) | – | Karşılaştırma noktası (zorunlu) |
| E02 | Contrast stretching | [0, p99.5] → [0, 255] | Kare maksimumu tek tük parlak piksellerle 195–255 arasında oynuyor (Bölüm 2.3); maksimum yerine yüzdelik kullanmak daha kararlı |
| E03 | Gamma | γ = 0.5 | Karanlık değerleri açar; soluk hücrelerin (ortanca parlaklık ≈14) sinyalini büyütür |
| E04 | Logaritmik | s = 255·log(1+r)/log(256) | Gamma'ya benzer ama karanlık uçta daha güçlü açar |
| E05 | Histogram eşitleme | – | Histogramı tüm aralığa yayar |
| E06 | CLAHE | clipLimit = 2, 8×8 blok | Yerel kontrast; 8×8 blok = 128 piksel ≈ 2 hücre çapı, düzensiz arka plana karşı |
| E07 | Histogram eşleme | Referans: geliştirme kümesinde p99'u ortanca olan kare | Kareler arası parlaklık farkını giderir; referans yalnızca geliştirme kümesinden |

In [ ]:
# Bölüm 6 (a): Yoğunluk dönüşümleri + kontrollü deney kayıtları
from skimage import exposure

# ---- Deney kayıt sistemi (Bölüm 10'daki experiments.csv bu listeden üretilecek) ----
DENEYLER = []

def dev_degerlendir(say_fonksiyonu):
    """Bir sayım fonksiyonunu 10 geliştirme karesinde çalıştırır: (tahminler, ortalama süre ms)."""
    tahminler, sureler = [], []
    for k, g in dev_goruntuler.items():
        t0 = time.perf_counter()
        tahminler.append(say_fonksiyonu(g))
        sureler.append(time.perf_counter() - t0)
    return tahminler, 1000 * np.mean(sureler)

def deney_ekle(deney_id, yogunluk, filtre, segmentasyon, morfoloji, parametreler, say_fonksiyonu):
    tahminler, sure_ms = dev_degerlendir(say_fonksiyonu)
    gercek = [dev_referans[k] for k in dev_goruntuler]
    kayit = {"deney_id": deney_id, "yogunluk": yogunluk, "filtre": filtre,
             "segmentasyon": segmentasyon, "morfoloji": morfoloji, "parametreler": parametreler,
             **metrikler(tahminler, gercek), "sure_ms": sure_ms,
             "tahminler": tahminler}
    DENEYLER[:] = [d for d in DENEYLER if d["deney_id"] != deney_id] + [kayit]   # tekrar çalıştırmada üzerine yaz
    return kayit

# ---- Yoğunluk dönüşümleri ----
def contrast_stretch(g, ust_yuzdelik=99.5):
    ust = np.percentile(g, ust_yuzdelik)
    return np.clip(g.astype(np.float32) * 255.0 / ust, 0, 255).astype(np.uint8)

def gamma_donusumu(g, gamma=0.5):
    return (255.0 * (g / 255.0) ** gamma).astype(np.uint8)

def log_donusumu(g):
    return (255.0 * np.log1p(g.astype(np.float32)) / np.log(256.0)).astype(np.uint8)

CLAHE = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))

# Histogram eşleme referansı: geliştirme kümesinde p99 değeri ortanca olan kare (yalnızca geliştirme kümesinden)
dev_p99 = {k: np.percentile(g, 99) for k, g in dev_goruntuler.items()}
HM_REFERANS_KARE = sorted(dev_p99, key=dev_p99.get)[len(dev_p99) // 2]
HM_REFERANS = dev_goruntuler[HM_REFERANS_KARE]

def histogram_esleme(g):
    return exposure.match_histograms(g, HM_REFERANS).astype(np.uint8)

YOGUNLUK = {
    "E01": ("Orijinal",           "-",                                 lambda g: g),
    "E02": ("Contrast stretching", "[0, p99.5] -> [0, 255]",           contrast_stretch),
    "E03": ("Gamma",              "gamma=0.5",                         gamma_donusumu),
    "E04": ("Logaritmik",         "255*log(1+r)/log(256)",             log_donusumu),
    "E05": ("Histogram eşitleme", "-",                                 cv2.equalizeHist),
    "E06": ("CLAHE",              "clipLimit=2, 8x8 blok",             CLAHE.apply),
    "E07": ("Histogram eşleme",   f"referans t{HM_REFERANS_KARE:03d}", histogram_esleme),
}
print(f"Histogram eşleme referansı: t{HM_REFERANS_KARE:03d} (p99 = {dev_p99[HM_REFERANS_KARE]:.0f})\n")

for deney_id, (ad, param, donusum) in YOGUNLUK.items():
    deney_ekle(deney_id, ad, "-", "Otsu", "-", f"{param}; min_alan={BASELINE_MIN_ALAN}",
               lambda g, f=donusum: baseline_segment(f(g), BASELINE_MIN_ALAN)[0])

# Otsu eşiğinin dönüşüme göre değişimi (her karenin eşiği)
otsu_esikleri = pd.DataFrame({ad: [cv2.threshold(f(g), 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)[0]
                                   for g in dev_goruntuler.values()]
                              for ad, _, f in YOGUNLUK.values()}, index=list(dev_goruntuler)).T

tablo6 = pd.DataFrame(DENEYLER).set_index("deney_id")
tablo6["otsu_esigi_ort"] = otsu_esikleri.mean(axis=1).values
tablo6["hatalar"] = [[int(t - dev_referans[k]) for t, k in zip(d["tahminler"], dev_goruntuler)]
                     for d in DENEYLER]
print(tablo6[["yogunluk", "parametreler", "MAE", "RMSE", "Bias", "Exact", "otsu_esigi_ort", "sure_ms"]]
      .round(2).to_string())
print("\nKare bazında hata (tahmin - referans), kareler:", list(dev_goruntuler))
for deney_id, hatalar in tablo6["hatalar"].items():
    print(f"  {deney_id}: {hatalar}")

In [ ]:
# Bölüm 6 (b): İşlem öncesi/sonrası görüntüler, histogramlar ve Otsu maskeleri
# Gösterilen kare: baseline'ın en zorlandığı geliştirme karesi (Bölüm 5'te en büyük hata).
k = en_kotu
g = dev_goruntuler[k]
fig, ax = plt.subplots(3, len(YOGUNLUK), figsize=(4 * len(YOGUNLUK), 12))
for i, (deney_id, (ad, param, donusum)) in enumerate(YOGUNLUK.items()):
    d = donusum(g)
    sayi, maske, _, esik = baseline_segment(d, BASELINE_MIN_ALAN)

    ax[0, i].imshow(d, cmap="gray", vmin=0, vmax=255)       # dönüşümün kendisi görülsün diye kontrast artırılmaz
    ax[0, i].set_title(f"{deney_id} {ad}\n{param}", fontsize=10)
    ax[0, i].axis("off")

    ax[1, i].bar(np.arange(256), np.bincount(d.ravel(), minlength=256), width=1.0)
    ax[1, i].axvline(esik, color="red", lw=1, label=f"Otsu = {esik:.0f}")
    ax[1, i].set_yscale("log"); ax[1, i].set_xlim(0, 255)
    ax[1, i].legend(fontsize=8); ax[1, i].set_title("Histogram (log)", fontsize=10)

    ax[2, i].imshow(maske, cmap="gray")
    ax[2, i].set_title(f"Otsu maskesi: {sayi} nesne (ref {dev_referans[k]})", fontsize=10)
    ax[2, i].axis("off")
fig.suptitle(f"t{k:03d} - yoğunluk dönüşümleri (görüntü / histogram / Otsu maskesi)", fontsize=14)
plt.tight_layout()
plt.show()